In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load clustered dataset
df = pd.read_csv(
    r"D:\hiPSC_Morphology_Project\data\processed\features_clustered.csv"
)

print(df.head())
print(df.shape)
print(df["cluster"].value_counts())

                                          image_name  label    area  \
0  00031b2f_3500000969_10X_20170613_11_27(10)-Sce...      1  4420.0   
1  00031b2f_3500000969_10X_20170613_11_27(10)-Sce...      2  1605.0   
2  00031b2f_3500000969_10X_20170613_11_27(10)-Sce...      3  1216.0   
3  00031b2f_3500000969_10X_20170613_11_27(10)-Sce...      4   549.0   
4  00031b2f_3500000969_10X_20170613_11_27(10)-Sce...      5  1412.0   

    perimeter  eccentricity  solidity    extent  major_axis_length  \
0  306.806133      0.878738  0.855098  0.532851         113.108218   
1  161.480231      0.778643  0.951393  0.597098          57.766271   
2  136.781746      0.271627  0.970471  0.643386          41.052407   
3   91.112698      0.779800  0.973404  0.769986          33.845157   
4  146.994949      0.672782  0.957938  0.626442          49.652234   

   minor_axis_length  convex_area  equivalent_diameter  cluster  
0          53.986993       5169.0            75.018123        0  
1          36.246447

In [5]:
feature_columns = [
    "area",
    "perimeter",
    "eccentricity",
    "solidity",
    "extent",
    "major_axis_length",
    "minor_axis_length",
    "convex_area",
    "equivalent_diameter"
]

cluster_means = df.groupby("cluster")[feature_columns].mean()

display(cluster_means)

,area,perimeter,eccentricity,solidity,extent,major_axis_length,minor_axis_length,convex_area,equivalent_diameter
cluster,,,,,,,,,
0,7504.790951,374.863877,0.759452,0.866498,0.579304,119.088385,64.600763,10563.129215,77.982438
1,338804.768212,7043.990699,0.763894,0.578284,0.394747,1061.159647,632.676349,567261.239073,623.336156


In [7]:
from scipy.stats import mannwhitneyu

results = []

for feature in feature_columns:

    c0 = df[df.cluster == 0][feature]
    c1 = df[df.cluster == 1][feature]

    stat, p = mannwhitneyu(c0, c1)

    results.append([feature, stat, p])

stats_df = pd.DataFrame(
    results,
    columns=["Feature", "Statistic", "p-value"]
)

display(stats_df)

,Feature,Statistic,p-value
0,area,35609.0,0.000000
1,perimeter,15673.0,0.000000
2,eccentricity,48185240.0,0.000019
3,solidity,96489839.0,0.000000
4,extent,89487288.5,0.000000
5,major_axis_length,78142.0,0.000000
6,minor_axis_length,86780.0,0.000000
7,convex_area,1608.0,0.000000
8,equivalent_diameter,35609.0,0.000000


In [8]:
def cohens_d(x, y):

    nx = len(x)
    ny = len(y)

    pooled_std = np.sqrt(
        ((nx - 1) * np.var(x, ddof=1) +
         (ny - 1) * np.var(y, ddof=1))
        / (nx + ny - 2)
    )

    return (np.mean(x) - np.mean(y)) / pooled_std


effect_sizes = []

for feature in feature_columns:

    c0 = df[df.cluster == 0][feature]
    c1 = df[df.cluster == 1][feature]

    d = cohens_d(c0, c1)

    effect_sizes.append([feature, d])

effect_df = pd.DataFrame(
    effect_sizes,
    columns=["Feature", "Cohen's d"]
)

display(effect_df)

,Feature,Cohen's d
0,area,-8.743727
1,perimeter,-9.976092
2,eccentricity,-0.029289
3,solidity,2.390172
4,extent,1.520313
5,major_axis_length,-8.088386
6,minor_axis_length,-8.662799
7,convex_area,-10.617292
8,equivalent_diameter,-8.290826
